# D1.9 · Detections whose subject is the agent platform

**Function D — AI for SecOps → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

Builds on **[D1.8 · Threat intel sub-lane](https://spbreed.github.io/cyber-commons/lessons/D1.8.html)**.

| | |
|---|---|
| Open-source tooling | Falco, Gitleaks, Sigstore |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The escape, the poisoned cache entry and the silently expired exemption all look like normal operation from inside the workload. These are detections whose subject is the platform running the agent, and not one of them reads the agent's own logs.

## 2 · The framework

```
   subject of the detection = the platform, not the workload

   escape primitives      ptrace non-child . LD_PRELOAD . /proc/self/mem
                          unshare/setns . non-harness write to the RPC path
                          a process that outlives its tool call

   cache integrity        repository vs manifest, hourly
                          modified | unexpected | MISSING

   credential exposure    repos, CI logs, image layers, public internet
                          -> automatic revocation, minutes not hours

   exemption drift        live control state vs the approved register
```

D1.4 wrote detections for what an agent does to your estate. These are
detections for what happens to **the platform running it** — and they are
different rules, on different data, with different owners.

Four of them come straight out of the incident register, and each catches
something that looks like normal operation from inside the workload:

**Escape detection (C1.4).** Named primitives, not anomaly scoring: `ptrace`
attach to a non-child, `LD_PRELOAD` on a spawned process, writes to
`/proc/self/mem`, unexpected `unshare`/`setns`, a new listening socket, and —
the one the incident's spoofing depended on — a background process that
outlives the tool call that spawned it. Plus any process that is not the
harness writing to the harness's RPC path.

**Cache integrity diffing (C5.4).** An hourly comparison of what is in the
artifact repository against a manifest of what should be. Poisoned targets were
staged in exactly this way, and a deletion is as much a signal as a
modification.

**Secret scanning wired to revocation (C4.1).** Not only source: CI logs, image
layers, wikis, ticket attachments, and external exposure monitoring — the
incident's credentials were found on the public internet. Revocation has to be
automatic, because the interval between discovery and redistribution was
minutes.

**Exemption-state reconciliation (C6.3).** Live control state against the
approved register, continuously. Any control disabled without a matching
unexpired exemption is a P1, and expiry re-enables automatically with no grace
period.

## 3 · Named primitives beat a generic anomaly score

In [ ]:
EVENTS = [
 {"t": 1,  "proc": "harness",   "syscall": "execve",  "arg": "python3 solve.py"},
 {"t": 2,  "proc": "solve.py",  "syscall": "openat",  "arg": "/workspace/in.json"},
 {"t": 3,  "proc": "solve.py",  "syscall": "ptrace",  "arg": "attach pid=1 (not a child)"},
 {"t": 4,  "proc": "solve.py",  "syscall": "execve",  "arg": "sh -c 'LD_PRELOAD=/tmp/x.so ...'"},
 {"t": 5,  "proc": "x.so",      "syscall": "openat",  "arg": "/run/harness.sock"},
 {"t": 6,  "proc": "solve.py",  "syscall": "openat",  "arg": "/workspace/out.json"},
 {"t": 7,  "proc": "sidecar",   "syscall": "execve",  "arg": "sleep 86400"},
]
BASELINE_TREE = {"harness", "solve.py"}

RULES = [
 ("ptrace attach to a non-child", lambda e: e["syscall"] == "ptrace" and "not a child" in e["arg"]),
 ("LD_PRELOAD on a spawned process", lambda e: "LD_PRELOAD" in e["arg"]),
 ("non-harness write to the RPC path", lambda e: "harness.sock" in e["arg"] and e["proc"] != "harness"),
 ("process outside the baseline tree", lambda e: e["proc"] not in BASELINE_TREE),
]

fired = []
for e in EVENTS:
    for name, rule in RULES:
        if rule(e):
            fired.append((e["t"], name, e["proc"]))
print(f"{'t':>3}  {'rule':36s}process")
for t, name, proc in fired:
    print(f"{t:>3}  {name:36s}{proc}")

# a generic score over the same events
volume = len(EVENTS)
score = volume / 100
print(f"\ngeneric anomaly score for this run: {score:.2f}  "
      f"(seven syscalls is unremarkable)")
print(f"named rules fired: {len(fired)}")
print()
print("The escape is four ordinary-looking events. Volume scoring cannot see it")
print("because nothing about the volume is unusual.")
assert len(fired) >= 5 and score < 0.1

## 4 · The one the spoofing depended on

In [ ]:
TOOL_CALLS = [
 {"call": 1, "spawned": ["solve.py"],  "ended": ["solve.py"]},
 {"call": 2, "spawned": ["sidecar"],   "ended": []},
 {"call": 3, "spawned": ["fmt"],       "ended": ["fmt"]},
]
def orphans(calls):
    live = []
    for c in calls:
        for p in c["spawned"]:
            if p not in c["ended"]:
                live.append((c["call"], p))
    return live

live = orphans(TOOL_CALLS)
print("processes surviving the tool call that spawned them:")
for call, proc in live:
    print(f"   call {call}: {proc}")
print()
print("A background process that outlives its tool call is how a substituted")
print("command stays in place for the next one. This single rule is the cheapest")
print("detection in the set and it is specific enough to page on.")
assert live == [(2, "sidecar")]

## 5 · Cache integrity, credential exposure, exemption drift

In [ ]:
import hashlib

def h(s): return hashlib.sha256(s.encode()).hexdigest()[:12]

MANIFEST = {"libtarget-1.4.jar": h("libtarget-1.4"),
            "runner-2.0.tar":    h("runner-2.0"),
            "fixtures-9.zip":    h("fixtures-9")}
REPOSITORY = {"libtarget-1.4.jar": h("libtarget-1.4-modified"),   # staged
              "runner-2.0.tar":    h("runner-2.0"),
              "extra-0.1.jar":     h("extra-0.1")}                # unexpected

def diff(manifest, repo):
    out = []
    for name in sorted(set(manifest) | set(repo)):
        if name not in repo:
            out.append((name, "MISSING - expected artifact removed"))
        elif name not in manifest:
            out.append((name, "UNEXPECTED - not in the manifest"))
        elif manifest[name] != repo[name]:
            out.append((name, "MODIFIED - hash mismatch"))
    return out

print("cache integrity diff (C5.4)")
for name, why in diff(MANIFEST, REPOSITORY):
    print(f"   {name:22s}{why}")

EXPOSURES = [("hf_liveToken", "public dataset card", 0)]
def revoke(exposures, human_in_loop):
    delay = 240 if human_in_loop else 2       # minutes
    return [(t, where, delay) for t, where, _ in exposures]

print("\ncredential exposure to revocation (C4.1)")
for mode, hitl in (("human in the loop", True), ("automated", False)):
    for token, where, delay in revoke(EXPOSURES, hitl):
        print(f"   {mode:20s}{token:16s}{where:22s}{delay:>4} min")
print("   reported interval from discovery to redistribution: minutes")

APPROVED = {"cyber-classifier": 30}
LIVE = {"cyber-classifier": "disabled", "egress-allowlist": "disabled"}
print("\nexemption-state reconciliation (C6.3), day 44")
for control, state in sorted(LIVE.items()):
    expires = APPROVED.get(control)
    if state == "enabled":
        verdict = "ok"
    elif expires is None:
        verdict = "P1 - disabled with no approved exemption"
    elif 44 > expires:
        verdict = f"P1 - exemption expired on day {expires}, auto re-enable"
    else:
        verdict = "ok - within an approved exemption"
    print(f"   {control:20s}{state:10s}{verdict}")
assert len(diff(MANIFEST, REPOSITORY)) == 3

## 6 · Verify — who owns each of these

In [ ]:
OWNERS = {
 "escape detection (C1.4)":        ("platform security", "host sensor"),
 "cache integrity (C5.4)":         ("build/release",     "artifact repository"),
 "secret scanning (C4.1)":         ("appsec",            "repos, CI, internet"),
 "exemption drift (C6.3)":         ("governance",        "control registry"),
 "upload scanning (C3.4)":         ("the ingest team",   "upload path"),
}
print(f"{'detection':34s}{'owner':20s}data source")
for d in sorted(OWNERS):
    owner, src = OWNERS[d]
    print(f"{d:34s}{owner:20s}{src}")

print()
print(f"{len(OWNERS)} detections, {len({o for o, _ in OWNERS.values()})} different owners,")
print("and not one of them reads the workload's own logs. That is the property")
print("that makes them useful after A2.8: they survive the subject of the")
print("investigation having had write access to everything else.")
assert len({o for o, _ in OWNERS.values()}) == 5

## What you just proved

Four named rules fire on a seven-event escape sequence that scores 0.07 on a generic volume anomaly. The orphaned-process rule isolates the one background process that outlived its tool call. The cache diff reports one modified, one unexpected and one missing artifact; automated revocation closes a credential in 2 minutes against 240 with a human in the loop; and exemption reconciliation raises a P1 for both an expired exemption and an unapproved one.

## Your turn

Pick the orphaned-process rule and write it for your own platform. It is one query, it has a low false-positive rate, and on most agent platforms nobody has ever run it.

---

**Next → [D1.10 · Fleet-level correlation: seeing a swarm](https://spbreed.github.io/cyber-commons/lessons/D1.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*